In [1]:
# this notebook will scale the data and
# train test split
import pandas as pd
import json
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "mlData"

symbol = "BTCUSDT"
tf = "5m"

src_path = folder_path / "clean" / f"{symbol}-{tf}-v9-cleaned.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
df.head()

,timestamp,label,ROC_3,ROC_5,ROC_10,MOM_3,RETURNS_1,DELTA_1,BUY_RATIO,DELTA_3,...,RANGE_RATIO,RSI_14,RSI_SLOPE,STOCH_K,CCI_5,DIST_HIGH_5,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS
0,1703922900000,-1,-0.001003,-0.001423,-0.002869,-1.138651,-0.000306,-5.157845,0.463912,-9.902054,...,1.000207,34.896828,-5.832012,20.279356,-89.920853,1.742961,0.443375,4.808903,0.443375,0.084416
1,1703923200000,1,-0.000723,-0.001671,-0.003321,-0.827558,-0.000331,-29.107430,0.357073,-27.175385,...,0.889447,33.052197,-4.091530,19.039312,-114.670479,1.843247,0.433472,5.091789,0.433472,0.078453
2,1703923500000,-1,0.000471,-0.000227,-0.003441,0.526516,0.001108,4.437141,0.527086,-29.828133,...,1.921415,43.773056,7.117867,89.324463,115.841896,0.229636,1.921415,3.493012,1.921415,0.354870
3,1703923800000,-1,0.000310,-0.000082,-0.002196,0.345665,-0.000467,-10.290833,0.403879,-34.961121,...,0.675743,40.804222,5.907394,67.871468,21.452227,0.675743,1.427506,2.232854,1.427506,0.389991
4,1703924100000,-1,-0.000861,-0.001497,-0.002755,-0.928440,-0.001500,-40.018299,0.299708,-45.871990,...,1.804351,33.051498,-0.000698,17.579941,-136.534500,2.000427,0.426685,2.532587,0.426685,0.144186


# Train-Test Split prep 
```
After Feature engineering
    │
    ├── Rolling Z-score (window=500) — 9 features:
        └── Group L:  DELTA_1, DELTA_3, BUY_RATIO
        └── Group J:  ATR_5, ATR_14
        └── Group K:  STOCH_K
        └── Group M:  DIST_HIGH_5, DIST_HIGH_10, RANGE_POS
    │
    ├── Else global Z-score (fit on train only)
            └── Group I:  ROC_3, ROC_5, ROC_10, MOM_3, RETURNS_1
            └── Group J:  ATR_RATIO, ATR_NORM_ROC, RANGE_RATIO
            └── Group K:  RSI_14, RSI_SLOPE, CCI_5
            └── Group L:  VOL_SPIKE, DELTA_DIV
            └── Group M:  DIST_LOW_5, DIST_LOW_10
    │
    └── All features
            └── winsorise at ±3σ
            └── drop first 500 rows (rolling warmup)
```

# Must be done in respective order:
- see [Train Data Pipeline](../../../../../journal/trainData-report/trainData-pipeline.md) for full data preparation
- start with df_clean
- Step-by-step
  1. Rolling Z-score (DELTA_1, DELTA_3, BUY_RATIO) — creates NaNs in first 500 rows
  2. Drop first 500 warmup rows — NOW the df is clean
  3. Temporal split into train/val/test — BEFORE any global fitting
  4. Global Z-score — fit on train only, transform val/test
  5. Winsorise — applied after Z-score, on each split separately

In [3]:
# CORRECT at inference — only past 500 bars available
def rolling_zscore_live(series, window=500):
    """
    For each bar t, compute mean and std using only bars t-499 through t.
    This is what rolling().mean() already does — just make it explicit
    in your inference pipeline so you never accidentally use future bars.
    """
    return (series - series.rolling(window).mean()) / \
            series.rolling(window).std()

In [4]:
# STEP:1 500 Rolling adjustment
ROLLING_WINDOW = 500   # applies to all 9 features
                       # DELTA_1, DELTA_3, BUY_RATIO
                       # ATR_5, ATR_14
                       # STOCH_K
                       # DIST_HIGH_5, DIST_HIGH_10, RANGE_POS

ROLLING_FEATURES = [
    # Group L
    "DELTA_1", "DELTA_3", "BUY_RATIO",
    # Group J
    "ATR_5", "ATR_14",
    # Group K
    "STOCH_K",
    # Group M
    "DIST_HIGH_5", "DIST_HIGH_10", "RANGE_POS",
]

GLOBAL_FEATURES = [
    # Group I
    "ROC_3", "ROC_5", "ROC_10", "MOM_3", "RETURNS_1",
    # Group J
    "ATR_RATIO", "ATR_NORM_ROC", "RANGE_RATIO",
    # Group K
    "RSI_14", "RSI_SLOPE", "CCI_5",
    # Group M
    "DIST_LOW_5", "DIST_LOW_10",
]

# apply rolling Z-score on only rolling features
for col in ROLLING_FEATURES:
    df[col] = rolling_zscore_live(df[col], window=ROLLING_WINDOW)

In [5]:
# df = df.drop(columns=["timestamp"])
print(df.shape)
print(df.columns)
df.head()

(229730, 26)
Index(['timestamp', 'label', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1',
       'DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV', 'ATR_5',
       'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_14',
       'RSI_SLOPE', 'STOCH_K', 'CCI_5', 'DIST_HIGH_5', 'DIST_LOW_5',
       'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS'],
      dtype='object')


,timestamp,label,ROC_3,ROC_5,ROC_10,MOM_3,RETURNS_1,DELTA_1,BUY_RATIO,DELTA_3,...,RANGE_RATIO,RSI_14,RSI_SLOPE,STOCH_K,CCI_5,DIST_HIGH_5,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS
0,1703922900000,-1,-0.001003,-0.001423,-0.002869,-1.138651,-0.000306,NaN,NaN,NaN,...,1.000207,34.896828,-5.832012,NaN,-89.920853,NaN,0.443375,NaN,0.443375,NaN
1,1703923200000,1,-0.000723,-0.001671,-0.003321,-0.827558,-0.000331,NaN,NaN,NaN,...,0.889447,33.052197,-4.091530,NaN,-114.670479,NaN,0.433472,NaN,0.433472,NaN
2,1703923500000,-1,0.000471,-0.000227,-0.003441,0.526516,0.001108,NaN,NaN,NaN,...,1.921415,43.773056,7.117867,NaN,115.841896,NaN,1.921415,NaN,1.921415,NaN
3,1703923800000,-1,0.000310,-0.000082,-0.002196,0.345665,-0.000467,NaN,NaN,NaN,...,0.675743,40.804222,5.907394,NaN,21.452227,NaN,1.427506,NaN,1.427506,NaN
4,1703924100000,-1,-0.000861,-0.001497,-0.002755,-0.928440,-0.001500,NaN,NaN,NaN,...,1.804351,33.051498,-0.000698,NaN,-136.534500,NaN,0.426685,NaN,0.426685,NaN


In [6]:
# Trim start-up NaN and must be chonologically ordered
col = df['DELTA_3'] # example column that was in rolling

# Find first and last valid label index
first_valid_idx = col.first_valid_index()
last_valid_idx  = col.last_valid_index()

df = df.loc[first_valid_idx:last_valid_idx].reset_index(drop=True)

print(f"Label NaN trimmed — head up to index {first_valid_idx}, tail after index {last_valid_idx}")
print(f"Rows after trim: {len(df):,}")

# Chronological order must still hold
assert df['timestamp'].is_monotonic_increasing, "Timestamp order broken after trim!"

# Assert no NaN remains anywhere — only contiguous edge NaN in label were expected
remaining_nan = df.isnull().sum()
remaining_nan = remaining_nan[remaining_nan > 0]
assert len(remaining_nan) == 0, f"Unexpected NaN still present:\n{remaining_nan.to_string()}"

print("Chronological order: OK")
print("All columns clean: no NaN remaining.")

Label NaN trimmed — head up to index 499, tail after index 229729
Rows after trim: 229,231
Chronological order: OK
All columns clean: no NaN remaining.


In [7]:
# preview
print(df.shape)
print(df.columns)
df.head()

(229231, 26)
Index(['timestamp', 'label', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1',
       'DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV', 'ATR_5',
       'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_14',
       'RSI_SLOPE', 'STOCH_K', 'CCI_5', 'DIST_HIGH_5', 'DIST_LOW_5',
       'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS'],
      dtype='object')


,timestamp,label,ROC_3,ROC_5,ROC_10,MOM_3,RETURNS_1,DELTA_1,BUY_RATIO,DELTA_3,...,RANGE_RATIO,RSI_14,RSI_SLOPE,STOCH_K,CCI_5,DIST_HIGH_5,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS
0,1704072600000,1,0.001685,0.003026,0.002919,1.000887,0.001696,0.670158,1.335879,0.174103,...,1.625065,63.908829,5.641647,1.365036,166.666672,-1.164539,2.565625,-1.386520,2.852350,1.512180
1,1704072900000,0,0.005637,0.005915,0.006955,3.197013,0.003578,4.042341,1.175509,2.331724,...,3.570640,72.846352,16.472961,0.985471,152.420135,-0.276062,4.275447,-0.693168,5.022140,1.151612
2,1704073200000,-1,0.004809,0.004798,0.005604,2.692953,-0.000469,1.508258,1.207332,2.992555,...,1.288434,70.379196,12.571678,0.767657,76.217010,0.086586,3.820108,-0.406468,4.464095,0.928937
3,1704073500000,-1,0.003120,0.005178,0.005245,1.770795,0.000012,0.246334,0.333758,2.773126,...,1.159551,70.406784,6.497955,0.774602,50.534561,0.120333,3.949591,-0.374959,4.613912,0.930670
4,1704073800000,-1,-0.001242,0.004031,0.005394,-0.710152,-0.000786,-0.039068,0.102169,0.821162,...,0.803755,66.060707,-6.785645,0.332519,-6.347267,0.782812,2.986331,0.152283,4.058754,0.586547


# Train, test, val split
Temporal split

In [8]:
TRAIN_RATIO     = 0.70
VAL_RATIO       = 0.15
# TEST_RATIO    = 0.15 # implicit — whatever remains


# Sort chronologically first — never shuffle
df = df.sort_values('timestamp').reset_index(drop=True)

def temporal_split(df: pd.DataFrame):
    """
    Strict temporal split — NO shuffling.
    Shuffling causes look-ahead leakage: future bars bleed into train sequences.

    Returns:
        train_df, val_df, test_df
    """
    n        = len(df)
    n_train  = int(n * TRAIN_RATIO)
    n_val    = int(n * VAL_RATIO)

    train_df = df.iloc[:n_train].copy()
    val_df   = df.iloc[n_train : n_train + n_val].copy()
    test_df  = df.iloc[n_train + n_val :].copy()

    print(f"Split sizes — train: {len(train_df):,}  "
          f"val: {len(val_df):,}  "
          f"test: {len(test_df):,}")

    return train_df, val_df, test_df

(train_df,val_df,test_df) = temporal_split(df)
print(f"Trainable — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

Split sizes — train: 160,461  val: 34,384  test: 34,386
Trainable — Train: 160,461 | Val: 34,384 | Test: 34,386


In [9]:
# delete df to preserved memory
del df

In [10]:
# apply scaler
from sklearn.preprocessing import StandardScaler
import joblib
from datetime import date

formatted_date = date.today().strftime("%Y%m")

ALL_FEATURES     = GLOBAL_FEATURES + ROLLING_FEATURES   # final column order

model_name = 'all-v9'

scaler = StandardScaler()
train_df[GLOBAL_FEATURES] = scaler.fit_transform(train_df[GLOBAL_FEATURES])  # fit + transform on train only
val_df[GLOBAL_FEATURES]   = scaler.transform(val_df[GLOBAL_FEATURES])        # transform only
test_df[GLOBAL_FEATURES]  = scaler.transform(test_df[GLOBAL_FEATURES])       # transform only

joblib.dump(scaler, folder_path / "scaler" / f"{formatted_date}-{model_name}-scaler.pkl")
print("Scaler saved.")


Scaler saved.


# Winsorise 
limited to +- 3SD

In [11]:
WINSOR_CLIP     = 3.0       # clip at ±3σ after Z-score

def winsorise(df: pd.DataFrame, clip: float = WINSOR_CLIP) -> pd.DataFrame:
    """
    Clip all features at ±clip standard deviations.
    Applied AFTER Z-scoring — values are already in σ units so
    
    It replaces the value, it does not remove the row.
    clipping at ±3 means replace > 3sd outlier with 3SD value

    Applied independently per split — no cross-split contamination.
    """
    df = df.copy()
    df[ALL_FEATURES] = df[ALL_FEATURES].clip(lower=-clip, upper=clip)
    return df

print("before: on train_df")
print(train_df.max())
train_df = winsorise(train_df, clip=WINSOR_CLIP)
val_df   = winsorise(val_df,   clip=WINSOR_CLIP)
test_df  = winsorise(test_df,  clip=WINSOR_CLIP)
print("after: on train_df")
print(train_df.max())

before: on train_df
timestamp       1.752211e+12
label           1.000000e+00
ROC_3           1.608470e+01
ROC_5           1.491731e+01
ROC_10          1.578876e+01
MOM_3           1.081686e+01
RETURNS_1       1.999784e+01
DELTA_1         1.988519e+01
BUY_RATIO       4.053552e+00
DELTA_3         1.532216e+01
VOL_SPIKE       4.678635e+00
DELTA_DIV       1.000000e+00
ATR_5           1.344226e+01
ATR_14          9.538544e+00
ATR_RATIO       1.061635e+01
ATR_NORM_ROC    8.699938e+00
RANGE_RATIO     1.660582e+01
RSI_14          4.123535e+00
RSI_SLOPE       5.502591e+00
STOCH_K         1.912465e+00
CCI_5           1.824468e+00
DIST_HIGH_5     8.431515e+00
DIST_LOW_5      8.671184e+00
DIST_HIGH_10    6.366370e+00
DIST_LOW_10     7.618922e+00
RANGE_POS       2.021611e+00
dtype: float64
after: on train_df
timestamp       1.752211e+12
label           1.000000e+00
ROC_3           3.000000e+00
ROC_5           3.000000e+00
ROC_10          3.000000e+00
MOM_3           3.000000e+00
RETURNS_1       3.

In [12]:
# save train data
# where Y = 0 around 26 cells, remove them during training
train_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-train.jsonl"
val_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-val.jsonl"
test_path  = folder_path / "trainData" / f"{formatted_date}-{model_name}-test.jsonl"

train_path.parent.mkdir(parents=True, exist_ok=True)

train_df.to_json(train_path, orient="records", lines=True)
val_df.to_json(val_path, orient="records", lines=True)
test_df.to_json(test_path,  orient="records", lines=True)

print(f"Saved train: {train_path.name}")
print(f"Saved val: {val_path.name}")
print(f"Saved test:  {test_path.name}")


Saved train: 202603-all-v9-train.jsonl
Saved val: 202603-all-v9-val.jsonl
Saved test:  202603-all-v9-test.jsonl


In [13]:
non_feature = ['timestamp', 'label']
feature_cols = [c for c in train_df.columns if c not in non_feature]

print(f"Train columns  : {len(train_df.columns)}  → {list(train_df.columns)}")

print(f"Scaled columns : {len(GLOBAL_FEATURES)}  → {GLOBAL_FEATURES}")


Train columns  : 26  → ['timestamp', 'label', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1', 'DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV', 'ATR_5', 'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_14', 'RSI_SLOPE', 'STOCH_K', 'CCI_5', 'DIST_HIGH_5', 'DIST_LOW_5', 'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS']
Scaled columns : 13  → ['ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_14', 'RSI_SLOPE', 'CCI_5', 'DIST_LOW_5', 'DIST_LOW_10']
